# Programacion Dinamica

In [1]:
import gymnasium as gym
import numpy as np
import copy

env = gym.make('CliffWalking-v1', render_mode = 'human') # Crear ambiente
env = env.unwrapped  # Tener acceso a P

# Reto 1
## Complete el algoritmo de Iteracion de la Politica

In [2]:
class PolicyIteration:
    """ Policy iteration algorithm """
    def __init__(self, env, theta, gamma):
        self.env = env
        self.v = [0] * self.env.observation_space.n  # valor inicial en cero
        self.pi = [[0.25, 0.25, 0.25, 0.25]
                   for i in range(self.env.observation_space.n)]  # Iniciar con una distribucion uniforme
        self.theta = theta  # Umbral de convergencia para la evaluacion de la estrategia
        self.gamma = gamma  # Factor de descuento

    def policy_evaluation(self):  # Evaluacion de la politica
        cnt = 1  # contador
        while 1:
            max_diff = 0
            new_v = [0] * self.env.observation_space.n 
            for s in range(self.env.observation_space.n):
                qsa_list = []  # Comenzar a clacular todas Q(s,a) en estado s
                for a in range(self.env.action_space.n):
                    qsa = 0
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        qsa += p * (r + self.gamma * self.v[next_state] *
                                    (1 - done))
                    qsa_list.append(self.pi[s][a] * qsa)
                new_v[s] = sum(qsa_list)  # Relacion entre funciones de valor estado y accion
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))
            self.v = new_v
            if max_diff < self.theta: break  # Satisface la condicion de convergencia
            cnt += 1
        print("Evaluation de la politica completada despues de %d pasos" % cnt)

    def policy_improvement(self):  # Mejora de la politica
        for s in range(self.env.observation_space.n):
            qsa_list = []
            for a in range(self.env.action_space.n):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] *
                                (1 - done))
                qsa_list.append(qsa)
            maxq = max(qsa_list)
            cntq = qsa_list.count(maxq)  # Calcular cuantas acciones obtuvieron el mayor valor Q
            # Las acciones comparten la misma probabilidad de suceder
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]
        print("Actualizacion de la politica completada")
        return self.pi

    def policy_iteration(self):  # Iteracion de la politica
        while 1:
            self.policy_evaluation()
            old_pi = copy.deepcopy(self.pi)  # Hacer una deepcopy de la politica
            new_pi = self.policy_improvement()
            if old_pi == new_pi: break

In [9]:
theta = 0.001
gamma = 0.9
agent = PolicyIteration(env, theta, gamma)
agent.policy_iteration()

Evaluation de la politica completada despues de 89 pasos
Actualizacion de la politica completada
Evaluation de la politica completada despues de 81 pasos
Actualizacion de la politica completada
Evaluation de la politica completada despues de 44 pasos
Actualizacion de la politica completada
Evaluation de la politica completada despues de 12 pasos
Actualizacion de la politica completada
Evaluation de la politica completada despues de 1 pasos
Actualizacion de la politica completada


In [13]:
env = gym.make('CliffWalking-v1', render_mode = 'human')
#Obtener estado inicial
state, _ = env.reset()

for step in range(100):
    #Escoger la mejor acción
    action = np.argmax(agent.pi[state])
    #Obtener s', r', donde e info
    state, reward, done, _ , _ = env.step(action) 
    if done:
        break

#Cerrar ambiente        
env.close()

In [14]:
def print_agent(agent, action_meaning, disaster=[], end=[]):
    print("Valor de paso：")
    for i in range(agent.env.shape[0]):
        for j in range(agent.env.shape[1]):
            print('%6.6s' % ('%.3f' % agent.v[i * agent.env.shape[1] + j]), end=' ')
        print()

    print("Estrategia：")
    for i in range(agent.env.shape[0]):
        for j in range(agent.env.shape[1]):
            if (i * agent.env.shape[1] + j) in disaster: # ubicacion del acantilado
                print('****', end=' ')
            elif (i * agent.env.shape[1] + j) in end:  # estado objetivo
                print('EEEE', end=' ')
            else:
                a = agent.pi[i * agent.env.shape[1] + j]
                pi_str = ''
                for k in range(len(action_meaning)):
                    pi_str += action_meaning[k] if a[k] > 0 else 'o'
                print(pi_str, end=' ')
        print()

In [15]:
action_meaning = ['^', '>', 'v', '<']
print_agent(agent, action_meaning, list(range(37, 47)), [47])

Valor de paso：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -1.000 -1.000 
Estrategia：
o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo oovo 
o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo oovo 
o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo oovo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 


# Reto 2
## Complete el Algoritmo de Iteracion de Valor

In [16]:
class ValueIteration:
    """ Algoritmo de Iteracion de Valor """
    def __init__(self, env, theta, gamma):
        self.env = env
        self.v = [0] * self.env.observation_space.n  # valor inicial es 0
        self.theta = theta  # Umbral de convergencia de valor
        self.gamma = gamma
        # La estrategia se obtiene despues que la iteracion de valor termina
        self.pi = [None for i in range(self.env.observation_space.n)]

    def value_iteration(self):
        cnt = 0
        while 1:
            max_diff = 0
            new_v = [0] * self.env.observation_space.n
            for s in range(self.env.observation_space.n):
                qsa_list = []  # Comenzar a calcular los valores Q(s,a) en estado s
                for a in range(env.action_space.n):
                    qsa = 0
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                    qsa_list.append(qsa)  
                new_v[s] = max(qsa_list)
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))
            self.v = new_v
            if max_diff < self.theta: break  # Satisface la condicion de convergencia
            cnt += 1
        print("Iteracion de valor tardo %d pasos" % cnt)
        self.get_policy()

    def get_policy(self):  # derivar la politica codiciosa basado en la funcion de valor
        for s in range(self.env.observation_space.n):
            qsa_list = []
            for a in range(self.env.action_space.n):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
            maxq = max(qsa_list)
            cntq = qsa_list.count(maxq)  # Calcular cuantas acciones obtuvieron el mayor valor para Q
            # Las acciones comparten la misma probabilidad de suceder
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]

In [18]:
env = gym.make('CliffWalking-v1') # Crear el ambiente
env = env.unwrapped  # Acceso a P

theta = 0.001
gamma = 0.9
agent = ValueIteration(env, theta, gamma)
agent.value_iteration()

Iteracion de valor tardo 14 pasos


In [20]:
env = gym.make('CliffWalking-v1', render_mode = 'human')
#Obtener estado inicial
state, _ = env.reset()

for step in range(100):
    #Escoger la mejor acción
    action = np.argmax(agent.pi[state])
    #Obtener s', r', donde e info
    state, reward, done, _ , _ = env.step(action) 
    if done:
        break

#Cerrar ambiente        
env.close()

In [21]:
print_agent(agent, action_meaning, list(range(37, 47)), [47])

Valor de paso：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -1.000 -1.000 
Estrategia：
o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo oovo 
o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo o>vo oovo 
o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo o>oo oovo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 
